# Make your columns file and settings file

A DECIMA run is steered by two small JSON files. They are different things, and mixing them up is the commonest early mistake:

| | **Columns file** | **Settings file** |
|---|---|---|
| Answers | *Which of my columns is which?* | *How should DECIMA compute?* |
| Content | DECIMA's column name → your column name | Every setting in `decima.Config` |
| Example | `{"teff": "TEFF", "Ks": "ks_m"}` | `{"n_keep": 1000, "extinction": "F19"}` |
| Make it | `decima columns stars.fits -o cols.json` | `decima default-config > settings.json` |
| Pass it | `--columns cols.json` or `columns={...}` | `--config settings.json` or `config=Config(...)` |
| Needed? | Unless your table already uses DECIMA's names | No: the defaults are the validated ones |

This page needs no models. It uses the example table from the [tutorial](../tutorial.ipynb).

In [1]:
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*IProgress.*")  # a harmless warning from a dependency

import decima
from astropy.table import Table

stars = Table.read("../data/example_stars.ecsv")
work = Path("../work")  # scratch folder for the files this page writes
work.mkdir(exist_ok=True)

## 1. Let DECIMA suggest the columns file

`suggest_columns` compares your column names with the usual names for each input: Gaia archive names, `E_X` / `X_err` / `X_error` error columns, `*_msigcom` for 2MASS, and so on. Case and punctuation are ignored.

In [2]:
suggestion = decima.suggest_columns(stars)
print(suggestion)

  ok  id                         <- 'GAIAEDR3_SOURCE_ID_1'
  ok  ra                         <- 'RAJ2000'
  ok  dec                        <- 'DEJ2000'
  ok  parallax                   <- 'PARALLAX'
  ok  parallax_error             <- 'E_PARALLAX'
  ok  teff                       <- 'TEFF'
  ok  teff_error                 <- 'E_TEFF'
  ok  feh                        <- 'FEH'
  ok  feh_error                  <- 'E_FEH'
  ok  alpha_fe                   <- 'ALPHA_FE'
  ok  alpha_fe_error             <- 'E_ALPHA_FE'
  ok  logg                       <- 'LOGG'
  ok  logg_error                 <- 'E_LOGG'
  ok  G                          <- 'GAIA_G'
  ok  BP                         <- 'GAIA_BP'
  ok  RP                         <- 'GAIA_RP'
  ok  J                          <- 'TWOMASS_J'
  ok  H                          <- 'TWOMASS_H'
  ok  Ks                         <- 'TWOMASS_K'
  ok  G_error                    <- 'E_GAIA_G'
  ok  BP_error                   <- 'E_GAIA_BP'
  ok  RP_error     

`suggestion` is a dict, which is the columns file. Save it, and read it back as the shell does:

In [3]:
(work / "cols.json").write_text(json.dumps(dict(suggestion), indent=2))
columns = json.loads((work / "cols.json").read_text())
list(columns.items())[:4]

[('id', 'GAIAEDR3_SOURCE_ID_1'),
 ('ra', 'RAJ2000'),
 ('dec', 'DEJ2000'),
 ('parallax', 'PARALLAX')]

## 2. When the suggestion is not enough

DECIMA never guesses between two candidates. Here a table has two T_eff columns, and no column that looks like [Fe/H]:

In [4]:
messy = Table({"TEFF": [5800.0], "teff_spec": [5750.0], "FEH_x": [0.0], "parallax": [2.0]})
s = decima.suggest_columns(messy)
print(s)
print("ambiguous:", s.ambiguous)
print("missing:  ", s.missing)

  --  ra                         NOT FOUND (required: Right ascension (ICRS), for the Bailer-Jones+21 distance prior.)
  --  dec                        NOT FOUND (required: Declination (ICRS), for the Bailer-Jones+21 distance prior.)
  ok  parallax                   (already named parallax)
  --  parallax_error             NOT FOUND (required: Uncertainty on parallax.)
  ??  teff                       several candidates ['TEFF', 'teff_spec']: choose one
  --  teff_error                 NOT FOUND (required: Uncertainty on T_eff. Used as given (no floor).); closest column: 'TEFF'
  --  feh                        NOT FOUND (required: Spectroscopic surface [Fe/H].); closest column: 'FEH_x'
  --  feh_error                  NOT FOUND (required: Uncertainty on [Fe/H]. Used as given (no floor).)
  --  Ks                         NOT FOUND (required: 2MASS Ks apparent magnitude (required: it gives the pseudo-magnitude).)
  --  Ks_error                   NOT FOUND (required: Uncertainty on 2MASS 

Decide yourself, and edit the dict. `ambiguous` lists the candidates, and for a missing column `close` names the closest column that was not used:

In [5]:
columns = dict(s)
columns["teff"] = "TEFF"        # chose one of the two candidates
columns["feh"] = s.close["feh"]  # 'FEH_x' really is [Fe/H]
columns

{'teff': 'TEFF', 'feh': 'FEH_x'}

## 3. Check the table, the columns file and the settings together

`check_inputs` (shell: `decima check stars.fits --columns cols.json --config settings.json`) is the step that saves hours. It reads your data and tells you what would go wrong. Here is a table with four planted mistakes (and one renamed column, which we map by hand):

In [6]:
import numpy as np

bad = Table.read("../data/example_stars.ecsv")
bad["TEFF"] = np.log10(bad["TEFF"])                       # log T_eff instead of K
bad["PARALLAX"] = bad["PARALLAX"] / 1000                  # arcseconds instead of mas
bad["TWOMASS_K"][1] = 99.0                                # 99 as "not measured"
bad["E_FEH"][2] = 0.0                                     # a zero error
bad.rename_column("FEH", "FEH_x")                         # a column DECIMA cannot find on its own
cols = dict(decima.suggest_columns(bad))
cols["feh"] = "FEH_x"
print(decima.check_inputs(bad, cols))

DECIMA input check: Table (7 rows)
Columns used:
  id                         <- 'GAIAEDR3_SOURCE_ID_1'
  ra                         <- 'RAJ2000'
  dec                        <- 'DEJ2000'
  parallax                   <- 'PARALLAX'
  parallax_error             <- 'E_PARALLAX'
  teff                       <- 'TEFF'
  teff_error                 <- 'E_TEFF'
  feh                        <- 'FEH_x'
  feh_error                  <- 'E_FEH'
  alpha_fe                   <- 'ALPHA_FE'
  alpha_fe_error             <- 'E_ALPHA_FE'
  logg                       <- 'LOGG'
  logg_error                 <- 'E_LOGG'
  G                          <- 'GAIA_G'
  BP                         <- 'GAIA_BP'
  RP                         <- 'GAIA_RP'
  J                          <- 'TWOMASS_J'
  H                          <- 'TWOMASS_H'
  Ks                         <- 'TWOMASS_K'
  G_error                    <- 'E_GAIA_G'
  BP_error                   <- 'E_GAIA_BP'
  RP_error                   <- 'E_GAIA_RP'
  J_erro

- **Errors** mean a run would stop (a required column is missing or misnamed, the settings are invalid).
- **Warnings** mean the run will go ahead on suspicious numbers. Fix them in your table; DECIMA does not change your data.
- The last lines say how many stars would be rejected, and why.

A mistake in the columns file itself is named the way you wrote it, with a suggestion:

In [7]:
print(decima.check_inputs(stars, {"teff": "TEF", "ks": "TWOMASS_K"}))

DECIMA input check: Table (7 rows)
Errors (a run would stop):
  x columns file: 'ks' is not one of DECIMA's column names (did you mean 'Ks'?). The file maps DECIMA's names to yours: {"teff": "YOUR_TEFF"}
  x columns file: teff -> 'TEF', but the table has no column 'TEF' (did you mean 'TEFF'?)
  x required column Ks (mag) is missing; 'TWOMASS_K' looks right: add "Ks": "TWOMASS_K" to the columns file (or run `decima columns`)
  x required column feh (dex) is missing; 'FEH' looks right: add "feh": "FEH" to the columns file (or run `decima columns`)
  x required column feh_error (dex) is missing; 'E_FEH' looks right: add "feh_error": "E_FEH" to the columns file (or run `decima columns`)
  x required column teff (K) is missing; 'TEFF' looks right: add "teff": "TEFF" to the columns file (or run `decima columns`)
  x required column teff_error (K) is missing; 'E_TEFF' looks right: add "teff_error": "E_TEFF" to the columns file (or run `decima columns`)
  x required column parallax (mas) is mi

## 4. The settings file

`decima default-config` prints the defaults. Save them, change what you want, keep the rest:

In [8]:
(work / "settings.json").write_text(decima.Config().to_json())
settings = json.loads((work / "settings.json").read_text())
settings["n_keep"] = 1000
settings["extinction"] = "F19_TEFF"
(work / "settings.json").write_text(json.dumps(settings, indent=2))

cfg = decima.Config.from_json((work / "settings.json").read_text())
cfg.n_keep, cfg.extinction

(1000, 'F19_TEFF')

A settings file may hold only the settings you change; the others keep their defaults. Mistakes are caught when the file is read, before any computation, and say how to fix them:

In [9]:
for text in ('{"n_thetas": 5}', '{"extinction": "F99"}', '{"device": "gpu"}', '{"mixture_alpha": 0.05}'):
    try:
        decima.Config.from_json(text)
    except ValueError as e:
        print(f"{text}\n    -> {str(e)[:170]}\n")

{"n_thetas": 5}
    -> unknown setting 'n_thetas' in the settings file (did you mean 'n_theta'?). Valid settings: n_latent_x, n_theta, n_keep, percentiles, seed, mixture_alpha, npe_ratio_ceilin

{"extinction": "F99"}
    -> unknown extinction law 'F99' (did you mean 'F19'?); choose from ('SS2018', 'WC2019', 'F19', 'F19_TEFF', 'none')

{"device": "gpu"}
    -> device must be 'cpu', 'cuda' or 'cuda:N', got 'gpu' (for a GPU use 'cuda')

{"mixture_alpha": 0.05}
    -> mixture_alpha * n_theta must be a whole number in [0, n_theta], got 0.05 * 10 = 0.5. With n_theta=10, mixture_alpha can be 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9,



## 5. Use them

```bash
decima check stars.fits --columns cols.json --config settings.json
decima run   stars.fits --columns cols.json --config settings.json --out run/
```

or in Python, `decima.infer("stars.fits", columns, cfg)`. The [tutorial](../tutorial.ipynb) does this end to end, and [Run a large catalogue](large-runs.md) covers `decima run`.

## Common problems

| You see | Cause and fix |
|---|---|
| `required column teff is missing; 'TEFF' looks right` | The table's name differs from DECIMA's: add `"teff": "TEFF"` to the columns file, or run `decima columns`. |
| `unknown canonical column ... did you mean 'teff'` | The columns file has the keys the wrong way round or misspelt: the key is **DECIMA's** name, the value is yours. |
| `column(s) not in the table: teff -> 'TEF'` | The value is not a column of your table. Names are case-sensitive. |
| `teff: median 3.77 ... Kelvin` | The column holds log₁₀ T_eff. Convert it to K in your table. |
| `Ks: 1 value(s) > 30 mag` | A flag such as 99 for "not measured". Replace it with NaN. |
| `parallax: 90% of |values| are below 0.02` | Parallax is in arcseconds. DECIMA wants mas. |
| `unknown setting 'n_thetas'` | A misspelt setting in the settings file; the message lists the valid ones. |